# AIHub T자형 교차로 데이터 준비 04 — 선별 영상 ZIP 만들기

`dedup_manifest_threshold12.csv`에서 `keep=True`인 대표 영상만 ZIP에 직접 넣습니다. 별도의 영상 복사 폴더를 만들지 않으며 원본 MP4도 삭제하지 않습니다.

ZIP을 Google Drive에 올리고 정상적으로 열리는 것을 확인한 후에 로컬 원본과 ZIP을 정리합니다.

In [ ]:
from datetime import datetime
from pathlib import Path
import json
import shutil
import zipfile

import pandas as pd
from tqdm.auto import tqdm

from src.stage2.labeler_widget import CSV_COLUMNS

VIDEO_ROOT = Path(
    '/mnt/c/ai허브/T자형교차로/095.교통사고_영상_데이터/'
    '01.데이터/1.Training/원천데이터_231108_add/'
    'TS_차대차_영상_T자형교차로'
)
REPO_ROOT = Path('/home/sungmin/blackbox-dacon')
MANIFEST_PATH = (
    REPO_ROOT / 'data/stage2/aihub_tjunction/metadata/dedup_manifest_threshold12.csv'
)
OUTPUT_ZIP = Path('/mnt/c/ai허브/T자형교차로/aihub_tjunction_curated_v1.zip')
PACKAGE_ROOT = 'aihub_tjunction_curated_v1'
RUN_FULL_CRC_CHECK = True

assert VIDEO_ROOT.is_dir(), VIDEO_ROOT
assert MANIFEST_PATH.is_file(), MANIFEST_PATH
OUTPUT_ZIP.parent.mkdir(parents=True, exist_ok=True)

print('영상 폴더:', VIDEO_ROOT)
print('결과 ZIP:', OUTPUT_ZIP)

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH, dtype={'video_id': str})
if manifest['keep'].dtype != bool:
    manifest['keep'] = manifest['keep'].astype(str).str.lower().eq('true')

selected = manifest[manifest['keep']].copy().reset_index(drop=True)
selected['source_path'] = selected['relative_path'].map(lambda value: VIDEO_ROOT / value)
selected['zip_video_path'] = selected['file_name'].map(
    lambda value: f'{PACKAGE_ROOT}/videos/{value}'
)

missing = selected.loc[~selected['source_path'].map(Path.is_file), 'source_path']
assert missing.empty, f'찾을 수 없는 영상이 있습니다: {missing.head().tolist()}'
assert selected['zip_video_path'].is_unique, 'ZIP 안에서 영상 파일명이 중복됩니다.'

selected_size_bytes = int(selected['source_path'].map(lambda path: path.stat().st_size).sum())
free_bytes = shutil.disk_usage(OUTPUT_ZIP.parent).free
required_bytes = selected_size_bytes + 512 * 1024 ** 2

summary = pd.Series({
    '선택 영상 수': len(selected),
    '예상 ZIP 크기(GB)': selected_size_bytes / 1024 ** 3,
    'C드라이브 여유 공간(GB)': free_bytes / 1024 ** 3,
})
display(summary.to_frame('값'))
assert free_bytes > required_bytes, 'ZIP 생성에 필요한 여유 공간이 부족합니다.'

## 라벨링용 CSV 준비

영상 한 번을 보면서 사용 가능 여부와 아직 학습 데이터가 없는 `entry_frame`, `entry_side`, `evasion_space`를 기록할 수 있는 빈 열을 추가합니다. `collision_frame`은 MM-AU 학습 데이터를 사용하므로 이번 AIHub 라벨링 범위에서 제외합니다.

In [ ]:
labeling = pd.DataFrame('', index=range(len(selected)), columns=CSV_COLUMNS)
labeling['ID'] = selected['video_id'].to_numpy()
labeling['video_path'] = selected['zip_video_path'].to_numpy()
labeling['fps'] = selected['fps'].to_numpy()
labeling['total_frames'] = selected['frame_count'].to_numpy()

display(labeling.head())

## ZIP 생성 및 검증

MP4는 이미 압축되어 있으므로 `ZIP_STORED`로 빠르게 묶습니다. 먼저 `.partial.zip`을 만들고 파일 수 및 CRC 검증이 끝난 경우에만 최종 이름으로 변경합니다.

In [ ]:
assert not OUTPUT_ZIP.exists(), f'결과 ZIP이 이미 존재합니다: {OUTPUT_ZIP}'
partial_zip = OUTPUT_ZIP.with_name(f'{OUTPUT_ZIP.stem}.partial.zip')
if partial_zip.exists():
    partial_zip.unlink()

manifest_csv = manifest.to_csv(index=False, encoding='utf-8-sig')
labeling_csv = labeling.to_csv(index=False, encoding='utf-8-sig')
package_info = {
    'package_name': PACKAGE_ROOT,
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'source_category': '차대차 영상 T자형교차로',
    'original_video_count': int(len(manifest)),
    'selected_video_count': int(len(selected)),
    'excluded_video_count': int(len(manifest) - len(selected)),
    'dedup_method': 'five-frame brightness-normalized pHash connected components',
    'phash_threshold': float(manifest['phash_threshold'].iloc[0]),
}

with zipfile.ZipFile(partial_zip, mode='w', compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
    archive.writestr(
        f'{PACKAGE_ROOT}/metadata/dedup_manifest_threshold12.csv',
        manifest_csv.encode('utf-8-sig'),
    )
    archive.writestr(
        f'{PACKAGE_ROOT}/metadata/stage2_labeling.csv',
        labeling_csv.encode('utf-8-sig'),
    )
    archive.writestr(
        f'{PACKAGE_ROOT}/metadata/package_info.json',
        json.dumps(package_info, ensure_ascii=False, indent=2).encode('utf-8'),
    )

    for row in tqdm(selected.itertuples(index=False), total=len(selected), desc='선별 영상 ZIP 생성'):
        archive.write(row.source_path, arcname=row.zip_video_path)

with zipfile.ZipFile(partial_zip, mode='r') as archive:
    video_entries = [name for name in archive.namelist() if name.startswith(f'{PACKAGE_ROOT}/videos/')]
    assert len(video_entries) == len(selected), (len(video_entries), len(selected))
    if RUN_FULL_CRC_CHECK:
        print('CRC 전체 검증 중...')
        bad_file = archive.testzip()
        assert bad_file is None, f'손상된 ZIP 항목: {bad_file}'

partial_zip.replace(OUTPUT_ZIP)

print('ZIP 생성 및 검증 완료:', OUTPUT_ZIP)
print(f'실제 ZIP 크기: {OUTPUT_ZIP.stat().st_size / 1024 ** 3:.2f} GB')
print(f'포함 영상 수: {len(selected):,}개')

## 로컬 파일 삭제 시점

1. 완성된 ZIP을 Google Drive에 업로드합니다.
2. Drive에서 ZIP 크기를 확인하고 Colab에서 파일 목록을 한 번 읽습니다.
3. 확인이 끝난 뒤에만 로컬의 원본 영상 폴더와 로컬 ZIP을 삭제합니다.

이 노트북은 원본을 자동 삭제하지 않습니다.